### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [13]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [14]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [15]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [16]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [17]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [18]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [19]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [20]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [21]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some key reasons in favor of choosing AutoGen for your AI Agent project:

1. **Natural Language Interactions**: AutoGen enables agents to negotiate through structured, multi-turn conversations using natural language. This reduces the need for complex API pipelines and makes interactions more intuitive.

2. **Simplified Workflow Management**: The framework helps in reducing the complexity involved in orchestrating multiple services, allowing for smoother hand-offs between different components of a project.

3. **Rapid Prototyping and Scaling**: With AutoGen, you can prototype complex workflows more easily, facilitating quicker iterations and expansions of your project as needed.

4. **Open-Source Flexibility**: Being an open-source solution, AutoGen offers flexibility for customization and integration, enabling developers to modify it to fit specific project needs.

5. **Minimized Bespoke Protocol Development**: By streamlining communication and operations, AutoGen reduces the need for designing and maintaining custom protocols, saving development time and effort.

These advantages can significantly enhance the efficiency and capability of your AI Agent project. 

TERMINATE

## Cons of AutoGen:
Here are some potential cons of using AutoGen in your AI Agent project:

1. **Complexity**: Implementing AutoGen can require a deep understanding of algorithmic prompts. This complexity can lead to longer development times and a steeper learning curve for the team.

2. **Less Structure**: Compared to some other frameworks, AutoGen may be perceived as less structured, which could complicate the implementation process and potentially lead to inconsistencies.

3. **Manual Efforts**: The reliance on manual configuration and customization might lead to higher workloads, especially for teams that are not familiar with the nuances of the framework.

4. **Limited Built-in Compliance**: AutoGen may not have the same level of built-in compliance and security features as other platforms, placing the onus on the team to ensure compliance and security.

5. **Performance Tuning**: Optimizing performance for specific use cases might require a considerable investment of time and resources, as the framework may not offer out-of-the-box solutions for all situations.

6. **Niche Suitability**: AutoGen is often better suited for research and experimental purposes, which might not align with the goals of all projects, especially those requiring a more straightforward and practical application.

These considerations should be weighed when deciding whether to adopt AutoGen for your project. 

TERMINATE



## Decision:

Based on the pros and cons presented, I recommend using AutoGen for the project. The advantages, particularly in enabling natural language interactions, simplifying workflow management, and providing rapid prototyping capabilities, align well with our goals of enhancing efficiency and fostering innovation in our AI Agent project. 

While the complexity and manual configuration requirements pose challenges, the benefits of flexibility and reduced development time outbalance these concerns. The open-source nature of AutoGen also provides opportunities for customization that could lead to a tailored solution fitting our project needs. 

Overall, the potential for increased capabilities and streamlined processes makes AutoGen a favorable choice for our team.

TERMINATE

In [22]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [23]:
await host.stop()